In [ ]:
from google.colab import files

uploaded = files.upload()


MessageError: RangeError: Maximum call stack size exceeded.

In [ ]:
import pandas as pd

df = pd.read_csv("main.csv")

print(df.head())


  campLocation         timestamp          source    destination       user  \
0    Bangalore  2023-09-13T01:24        10.1.1.1    172.16.0.20      guest   
1    Bangalore  2023-09-17T23:36        10.1.1.1    172.16.0.20  anonymous   
2    Bangalore  2023-09-18T09:47  84.124.148.119  48.239.159.33      admin   
3    Bangalore  2023-09-15T23:45    192.168.10.5       10.1.1.2  anonymous   
4    Bangalore  2023-09-16T07:17     192.168.0.1   172.16.1.101      admin   

           device             eventType  \
0       ServerABC     malware-detection   
1  Workstation123  network-disconnected   
2  Workstation123            api-called   
3       ServerABC          auth-success   
4       ServerABC           dns-queries   

                                    eventDescription  eventSeverity  \
0  Malware detected: File 'file910.txt' found on ...        warning   
1  Device 'Workstation123' disconnected from the ...          error   
2            API called: API2 from IP 84.124.148.119       

In [ ]:
!pip install cryptography pandas numpy scikit-learn xgboost tensorflow matplotlib


In [ ]:
import pandas as pd

csv_filename = "main.csv"
df = pd.read_csv(csv_filename)

print(df.head())


  campLocation         timestamp          source    destination       user  \
0    Bangalore  2023-09-13T01:24        10.1.1.1    172.16.0.20      guest   
1    Bangalore  2023-09-17T23:36        10.1.1.1    172.16.0.20  anonymous   
2    Bangalore  2023-09-18T09:47  84.124.148.119  48.239.159.33      admin   
3    Bangalore  2023-09-15T23:45    192.168.10.5       10.1.1.2  anonymous   
4    Bangalore  2023-09-16T07:17     192.168.0.1   172.16.1.101      admin   

           device             eventType  \
0       ServerABC     malware-detection   
1  Workstation123  network-disconnected   
2  Workstation123            api-called   
3       ServerABC          auth-success   
4       ServerABC           dns-queries   

                                    eventDescription  eventSeverity  \
0  Malware detected: File 'file910.txt' found on ...        warning   
1  Device 'Workstation123' disconnected from the ...          error   
2            API called: API2 from IP 84.124.148.119       

In [ ]:
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import serialization, hashes
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
import os
import base64

# Generate RSA Key Pair
def generate_rsa_keypair(key_size):
    private_key = rsa.generate_private_key(public_exponent=65537, key_size=key_size)
    public_key = private_key.public_key()
    return private_key, public_key

# Generate Weak (512-bit) and Strong (2048-bit) RSA Keys
weak_private, weak_public = generate_rsa_keypair(1024)
strong_private, strong_public = generate_rsa_keypair(2048)


In [ ]:
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
import numpy as np
import os

# AES Encryption
def aes_encrypt(data):
    key = os.urandom(32)  # 256-bit AES key
    iv = os.urandom(16)   # Initialization vector
    cipher = Cipher(algorithms.AES(key), modes.CBC(iv))
    encryptor = cipher.encryptor()

    # Pad data to be multiple of block size (16 bytes)
    pad_length = 16 - (len(data) % 16)
    padded_data = data + chr(pad_length) * pad_length
    ciphertext = encryptor.update(padded_data.encode()) + encryptor.finalize()

    return base64.b64encode(ciphertext).decode(), key, iv

# Encrypt AES Key using RSA
def encrypt_aes_key(aes_key, public_key):
    return base64.b64encode(public_key.encrypt(
        aes_key,
        padding.OAEP(mgf=padding.MGF1(hashes.SHA256()), algorithm=hashes.SHA256(), label=None)
    )).decode()

# Encrypt Full Row
def encrypt_row(row, public_key):
    row_data = " | ".join(map(str, row))  # Convert row to string
    aes_ciphertext, aes_key, aes_iv = aes_encrypt(row_data)  # Encrypt row with AES
    encrypted_key = encrypt_aes_key(aes_key, public_key)  # Encrypt AES key with RSA
    return aes_ciphertext, encrypted_key, base64.b64encode(aes_iv).decode()

# Apply Encryption to Each Row
df["KeyType"] = np.random.choice(["Weak", "Strong"], size=len(df))
df[["EncryptedLog", "EncryptedAESKey", "IV"]] = df.apply(
    lambda row: encrypt_row(row, weak_public if row["KeyType"] == "Weak" else strong_public),
    axis=1, result_type="expand"
)

print(df.head())


  campLocation         timestamp          source    destination       user  \
0    Bangalore  2023-09-13T01:24        10.1.1.1    172.16.0.20      guest   
1    Bangalore  2023-09-17T23:36        10.1.1.1    172.16.0.20  anonymous   
2    Bangalore  2023-09-18T09:47  84.124.148.119  48.239.159.33      admin   
3    Bangalore  2023-09-15T23:45    192.168.10.5       10.1.1.2  anonymous   
4    Bangalore  2023-09-16T07:17     192.168.0.1   172.16.1.101      admin   

           device             eventType  \
0       ServerABC     malware-detection   
1  Workstation123  network-disconnected   
2  Workstation123            api-called   
3       ServerABC          auth-success   
4       ServerABC           dns-queries   

                                    eventDescription  eventSeverity  \
0  Malware detected: File 'file910.txt' found on ...        warning   
1  Device 'Workstation123' disconnected from the ...          error   
2            API called: API2 from IP 84.124.148.119       

In [ ]:
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
import numpy as np

# AES Encryption
def aes_encrypt(data):
    key = os.urandom(32)  # 256-bit AES key
    iv = os.urandom(16)   # Initialization vector
    cipher = Cipher(algorithms.AES(key), modes.CBC(iv))
    encryptor = cipher.encryptor()

    # Pad data to be multiple of block size (16 bytes)
    pad_length = 16 - (len(data) % 16)
    padded_data = data + chr(pad_length) * pad_length
    ciphertext = encryptor.update(padded_data.encode()) + encryptor.finalize()

    return base64.b64encode(ciphertext).decode(), key, iv

# Encrypt AES Key using RSA
def encrypt_aes_key(aes_key, public_key):
    return base64.b64encode(public_key.encrypt(
        aes_key,
        padding.OAEP(mgf=padding.MGF1(hashes.SHA256()), algorithm=hashes.SHA256(), label=None)
    )).decode()

# Encrypt Full Row
def encrypt_row(row, public_key):
    row_data = " | ".join(map(str, row))  # Convert row to string
    aes_ciphertext, aes_key, aes_iv = aes_encrypt(row_data)  # Encrypt row with AES
    encrypted_key = encrypt_aes_key(aes_key, public_key)  # Encrypt AES key with RSA
    return aes_ciphertext, encrypted_key, base64.b64encode(aes_iv).decode()

# Apply Encryption to Each Row
df["KeyType"] = np.random.choice(["Weak", "Strong"], size=len(df))
df[["EncryptedLog", "EncryptedAESKey", "IV"]] = df.apply(
    lambda row: encrypt_row(row, weak_public if row["KeyType"] == "Weak" else strong_public),
    axis=1, result_type="expand"
)

print(df.head())


  campLocation         timestamp          source    destination       user  \
0    Bangalore  2023-09-13T01:24        10.1.1.1    172.16.0.20      guest   
1    Bangalore  2023-09-17T23:36        10.1.1.1    172.16.0.20  anonymous   
2    Bangalore  2023-09-18T09:47  84.124.148.119  48.239.159.33      admin   
3    Bangalore  2023-09-15T23:45    192.168.10.5       10.1.1.2  anonymous   
4    Bangalore  2023-09-16T07:17     192.168.0.1   172.16.1.101      admin   

           device             eventType  \
0       ServerABC     malware-detection   
1  Workstation123  network-disconnected   
2  Workstation123            api-called   
3       ServerABC          auth-success   
4       ServerABC           dns-queries   

                                    eventDescription  eventSeverity  \
0  Malware detected: File 'file910.txt' found on ...        warning   
1  Device 'Workstation123' disconnected from the ...          error   
2            API called: API2 from IP 84.124.148.119       

In [ ]:
# RSA Decrypt AES Key
def decrypt_aes_key(encrypted_key, private_key):
    return private_key.decrypt(
        base64.b64decode(encrypted_key),
        padding.OAEP(mgf=padding.MGF1(hashes.SHA256()), algorithm=hashes.SHA256(), label=None)
    )

# AES Decrypt
def aes_decrypt(ciphertext, key, iv):
    cipher = Cipher(algorithms.AES(key), modes.CBC(base64.b64decode(iv)))
    decryptor = cipher.decryptor()
    decrypted_data = decryptor.update(base64.b64decode(ciphertext)) + decryptor.finalize()
    return decrypted_data.rstrip(decrypted_data[-1:]).decode()

# Decrypt a Single Row
def decrypt_row(row):
    aes_key = decrypt_aes_key(row["EncryptedAESKey"], weak_private if row["KeyType"] == "Weak" else strong_private)
    decrypted_log = aes_decrypt(row["EncryptedLog"], aes_key, row["IV"])
    return decrypted_log

# Apply Decryption
df["DecryptedLog"] = df.apply(decrypt_row, axis=1)
print(df[["DecryptedLog"]].head())


                                        DecryptedLog
0  Bangalore | 2023-09-13T01:24 | 10.1.1.1 | 172....
1  Bangalore | 2023-09-17T23:36 | 10.1.1.1 | 172....
2  Bangalore | 2023-09-18T09:47 | 84.124.148.119 ...
3  Bangalore | 2023-09-15T23:45 | 192.168.10.5 | ...
4  Bangalore | 2023-09-16T07:17 | 192.168.0.1 | 1...


In [ ]:
# Shannon Entropy
def shannon_entropy(data):
    prob = [float(data.count(c)) / len(data) for c in set(data)]
    return -sum([p * np.log2(p) for p in prob])

df["Entropy"] = df["EncryptedLog"].apply(shannon_entropy)
df["Label"] = df["KeyType"].apply(lambda x: 0 if x == "Weak" else 1)

X = df[["Entropy"]]
y = df["Label"]


In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
from sklearn.pipeline import Pipeline as ImbPipeline


# 📌 2️⃣ Feature Extraction Functions
def shannon_entropy(data):
    """Compute Shannon entropy."""
    prob = [float(data.count(c)) / len(data) for c in set(data)]
    return -sum([p * np.log2(p) for p in prob])

def byte_distribution(data):
    """Compute frequency of each byte value (0-255)."""
    byte_counts = np.zeros(256)
    for byte in data.encode():
        byte_counts[byte] += 1
    return np.mean(byte_counts)  # Use mean to get a single value

def bit_pattern(data):
    """Compute number of ‘1’s in binary representation."""
    return sum(bin(byte).count('1') for byte in data.encode()) / len(data)

# 📌 3️⃣ Extract Features
df["Entropy"] = df["EncryptedLog"].apply(shannon_entropy)
df["ByteDist"] = df["EncryptedLog"].apply(byte_distribution)
df["BitPattern"] = df["EncryptedLog"].apply(bit_pattern)
df["Label"] = df["KeyType"].apply(lambda x: 0 if x == "Weak" else 1)

# 📌 4️⃣ Define Features & Labels
X = df[["Entropy", "ByteDist", "BitPattern"]]
y = df["Label"]

# 📌 5️⃣ Train-Test Split (with stratification)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# 📌 6️⃣ Scale Features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 📌 7️⃣ Handle Class Imbalance with SMOTE
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

# 📌 8️⃣ Define Ensemble Model
rf = RandomForestClassifier(n_estimators=200, max_depth=20, random_state=42)
gb = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, random_state=42)
xgb = XGBClassifier(n_estimators=200, learning_rate=0.05, random_state=42, eval_metric='mlogloss')

ensemble_model = StackingClassifier(
    estimators=[('rf', rf), ('gb', gb), ('xgb', xgb)],
    final_estimator=VotingClassifier(estimators=[('rf', rf), ('gb', gb), ('xgb', xgb)], voting='soft')
)

pipeline = ImbPipeline([
    ('classifier', ensemble_model)
])

# 📌 9️⃣ Train Model
pipeline.fit(X_train_resampled, y_train_resampled)

# 📌 🔟 Evaluate Model
y_pred = pipeline.predict(X_test_scaled)
print("Classification Report:")
print(classification_report(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       0.53      0.52      0.53      2023
           1       0.52      0.52      0.52      1977

    accuracy                           0.52      4000
   macro avg       0.52      0.52      0.52      4000
weighted avg       0.52      0.52      0.52      4000



ML MODEL